# Data cleaning through interpolation #

Created this notebook since I had errors in running interpolation.py, and doing it in notebook form allows for better readability (personal preference.

In [1]:
import pandas as pd
from datetime import datetime
from xarray import open_mfdataset
import numpy as np
from scipy.spatial import cKDTree

In [2]:
def convert_date_to_day_of_year(date_str):
    # Parse the input date string into a datetime object
    date_obj = datetime.strptime(date_str, "%Y-%m-%d")

    # Format the year and the day of the year into the desired format
    # %j in strftime gives the day of the year [001,366]
    converted_date = date_obj.strftime("%Y_%j")

    return converted_date

In [3]:
def convert_day_of_year_to_date(year_day_str):
    # Parse the input string into a datetime object
    # The format '%Y_%j' parses 'year_day of the year'
    date_obj = datetime.strptime(year_day_str, "%Y_%j")

    # Format the datetime object into a standard date string 'YYYY-MM-DD'
    standard_date = date_obj.strftime("%Y-%m-%d")

    return standard_date

In [4]:
def idw_interpolation(data, lat, lon, n_points=8):
    """
    Interpolates soil moisture for a given latitude and longitude using Inverse Distance Weighting (IDW).

    :param data: DataFrame containing the soil moisture data.
    :param lat: Latitude for which soil moisture is to be interpolated.
    :param lon: Longitude for which soil moisture is to be interpolated.
    :param n_points: Number of nearest points to consider for interpolation.
    :return: Interpolated soil moisture value.
    """
    # Build a KDTree for efficient nearest neighbor search
    tree = cKDTree(data[['Latitude', 'Longitude']])

    # Find the indices of the n_points nearest neighbors
    distances, indices = tree.query([lat, lon], k=n_points)

    # Avoid division by zero in case the exact point is in the dataset
    distances[distances == 0] = np.finfo(float).eps

    # Calculate weights based on inverse distance
    weights = 1 / distances
    
    if all(idx == 0 for idx in indices):
        return np.nan

    print("Data points used for interpolation:")
    print(data.iloc[indices])
    # Calculate weighted average of soil moisture
    weighted_moisture = np.sum(weights * data.iloc[indices]['Soil_Moisture']) / np.sum(weights)

    return float(weighted_moisture)

In [5]:
def read_netcdf_file(dataset):
    latitudes = dataset['latitude'].values
    longitudes = dataset['longitude'].values
    global_sm = dataset['SM_daily'].values[0]
    latitudes_flat = latitudes.flatten()
    longitudes_flat = longitudes.flatten()
    soil_moisture_flat = global_sm.flatten()

    # Create a DataFrame
    df = pd.DataFrame({
        'Latitude': latitudes_flat,
        'Longitude': longitudes_flat,
        'Soil_Moisture': soil_moisture_flat
    })
    df = df.dropna(subset=['Soil_Moisture'])
    return df

In [6]:
csv_user_address = "./satellite_data_csv/"
netcdf_user_address = "./complete_satellite_data/ucar_cu_cygnss_sm_v1_"

csv_filenames = ["Station1_Satellite.csv", "Station2_Satellite.csv","Station3_Satellite.csv","Station4_Satellite.csv",
                 "Station5_Satellite.csv", "Station6_Satellite.csv",]
stations_data = [
    {'Latitude': 30.3989, 'Longitude': -98.6105, 'Station_ID': '5'},
    {'Latitude': 30.4193, 'Longitude': -98.8046, 'Station_ID': '1'},
    {'Latitude': 30.4421, 'Longitude': -98.8427, 'Station_ID': '6'},
    {'Latitude': 30.4600, 'Longitude': -98.9407, 'Station_ID': '4'},
    {'Latitude': 30.2454, 'Longitude': -98.7059, 'Station_ID': '2'},
    {'Latitude': 30.2758, 'Longitude': -98.7242, 'Station_ID': '3'}
]

null_dates_dict = {}
for i in range(0, len(csv_filenames)):
    dataframe = pd.read_csv(csv_user_address + csv_filenames[i])
    for index, row in dataframe.iterrows():
        if pd.isnull(row['soil_moisture']):
            date = convert_date_to_day_of_year(row['Date'])
            if date in null_dates_dict:
                null_dates_dict[date].append(i)
            else:
                null_dates_dict[date] = [i]
interpolation_dataframe_list = []

for i in range(0,6):
    interpolation_dataframe_list.append(pd.DataFrame())



for date in null_dates_dict:
    file_path = netcdf_user_address + date + ".dap.nc4"
    print(file_path)
    if file_path == "./complete_satellite_data/ucar_cu_cygnss_sm_v1_2019_280.dap.nc4":
        continue
    try:
        dataset_file = open_mfdataset(file_path, combine="by_coords", parallel=True)
        dataset = read_netcdf_file(dataset_file)
        converted_date = convert_day_of_year_to_date(date)

        for station_number in null_dates_dict[date]:
            lat = stations_data[station_number]['Latitude']
            lon = stations_data[station_number]['Longitude']

            estimated_moisture = idw_interpolation(dataset, lat, lon)
            
            if np.isnan(estimated_moisture):
                print(converted_date, "could not interpolate")
            else:
                print(converted_date, estimated_moisture, "Station Number:", station_number)
            cur_dataframe = interpolation_dataframe_list[station_number]

            # Create a new DataFrame from the new_row dictionary
            new_row_df = pd.DataFrame([{'Date': converted_date, 'soil_moisture': estimated_moisture, 'distance': 0}])

            # Concatenate the new_row_df DataFrame with cur_dataframe
            interpolation_dataframe_list[station_number] = pd.concat([cur_dataframe, new_row_df], ignore_index=True)
    except OSError:
        print("file not found")

./complete_satellite_data/ucar_cu_cygnss_sm_v1_2017_077.dap.nc4
Data points used for interpolation:
         Latitude  Longitude  Soil_Moisture
182152  29.986300 -98.402489       0.228371
184558  30.966091 -98.402489       0.207799
183757  30.638416 -98.029045       0.182502
182153  29.986300 -98.029045       0.263171
181349  29.661814 -98.775932       0.267734
181350  29.661814 -98.402489       0.278897
184559  30.966091 -98.029045       0.206753
185359  31.294872 -98.775932       0.154644
2017-03-18 0.2243239324397684 Station Number: 0
Data points used for interpolation:
         Latitude  Longitude  Soil_Moisture
182152  29.986300 -98.402489       0.228371
184558  30.966091 -98.402489       0.207799
181349  29.661814 -98.775932       0.267734
183757  30.638416 -98.029045       0.182502
181348  29.661814 -99.149376       0.240971
181350  29.661814 -98.402489       0.278897
185359  31.294872 -98.775932       0.154644
182153  29.986300 -98.029045       0.263171
2017-03-18 0.22777054582

Data points used for interpolation:
         Latitude  Longitude  Soil_Moisture
183755  30.638416 -98.775932       0.149499
183754  30.638416 -99.149376       0.170931
184557  30.966091 -98.775932       0.165303
184556  30.966091 -99.149376       0.191267
183756  30.638416 -98.402489       0.187199
182951  30.311827 -99.522820       0.246567
183753  30.638416 -99.522820       0.202131
184558  30.966091 -98.402489       0.265416
2017-03-30 0.18613888313721216 Station Number: 3
Data points used for interpolation:
         Latitude  Longitude  Soil_Moisture
183755  30.638416 -98.775932       0.149499
183756  30.638416 -98.402489       0.187199
181349  29.661814 -98.775932       0.280124
183754  30.638416 -99.149376       0.170931
181350  29.661814 -98.402489       0.261940
184557  30.966091 -98.775932       0.165303
184558  30.966091 -98.402489       0.265416
183757  30.638416 -98.029045       0.235561
2017-03-30 0.20825376134559756 Station Number: 4
Data points used for interpolation:
  

Data points used for interpolation:
         Latitude  Longitude  Soil_Moisture
182954  30.311827 -98.402489       0.176588
183755  30.638416 -98.775932       0.131243
183756  30.638416 -98.402489       0.166871
182151  29.986300 -98.775932       0.182818
182152  29.986300 -98.402489       0.192870
182952  30.311827 -99.149376       0.177886
182955  30.311827 -98.029045       0.180323
183754  30.638416 -99.149376       0.126353
2017-04-20 0.16625106732132217 Station Number: 0
Data points used for interpolation:
         Latitude  Longitude  Soil_Moisture
183755  30.638416 -98.775932       0.131243
182952  30.311827 -99.149376       0.177886
183754  30.638416 -99.149376       0.126353
182954  30.311827 -98.402489       0.176588
182151  29.986300 -98.775932       0.182818
183756  30.638416 -98.402489       0.166871
184557  30.966091 -98.775932       0.159614
182150  29.986300 -99.149376       0.210701
2017-04-20 0.16174513903170956 Station Number: 1
Data points used for interpolation:
  

Data points used for interpolation:
         Latitude  Longitude  Soil_Moisture
183755  30.638416 -98.775932       0.141850
183754  30.638416 -99.149376       0.148685
182954  30.311827 -98.402489       0.174753
182151  29.986300 -98.775932       0.202885
183756  30.638416 -98.402489       0.213486
184557  30.966091 -98.775932       0.160958
182150  29.986300 -99.149376       0.210074
184556  30.966091 -99.149376       0.176248
2017-05-17 0.17226252934510855 Station Number: 2
Data points used for interpolation:
         Latitude  Longitude  Soil_Moisture
183755  30.638416 -98.775932       0.141850
183754  30.638416 -99.149376       0.148685
182151  29.986300 -98.775932       0.202885
182150  29.986300 -99.149376       0.210074
184557  30.966091 -98.775932       0.160958
184556  30.966091 -99.149376       0.176248
182954  30.311827 -98.402489       0.174753
183756  30.638416 -98.402489       0.213486
2017-05-17 0.17153445020247474 Station Number: 3
Data points used for interpolation:
  

Data points used for interpolation:
         Latitude  Longitude  Soil_Moisture
182954  30.311827 -98.402489       0.169542
183755  30.638416 -98.775932       0.132010
183756  30.638416 -98.402489       0.160608
182151  29.986300 -98.775932       0.168877
182152  29.986300 -98.402489       0.179308
182955  30.311827 -98.029045       0.208760
183754  30.638416 -99.149376       0.132453
183757  30.638416 -98.029045       0.193627
2017-06-11 0.1648731230845216 Station Number: 0
Data points used for interpolation:
         Latitude  Longitude  Soil_Moisture
183755  30.638416 -98.775932       0.132010
183754  30.638416 -99.149376       0.132453
182954  30.311827 -98.402489       0.169542
182151  29.986300 -98.775932       0.168877
183756  30.638416 -98.402489       0.160608
182152  29.986300 -98.402489       0.179308
184556  30.966091 -99.149376       0.150980
183753  30.638416 -99.522820       0.158326
2017-06-11 0.15279099890577771 Station Number: 1
Data points used for interpolation:
   

...

Data points used for interpolation:
         Latitude  Longitude  Soil_Moisture
182954  30.311827 -98.402489       0.145642
183755  30.638416 -98.775932       0.110767
183756  30.638416 -98.402489       0.153065
182150  29.986300 -99.149376       0.177831
183754  30.638416 -99.149376       0.130552
181349  29.661814 -98.775932       0.174788
184557  30.966091 -98.775932       0.121546
181350  29.661814 -98.402489       0.201484
2022-09-21 0.14866371787497262 Station Number: 5
./complete_satellite_data/ucar_cu_cygnss_sm_v1_2022_270.dap.nc4
Data points used for interpolation:
         Latitude  Longitude  Soil_Moisture
183755  30.638416 -98.775932       0.132745
183756  30.638416 -98.402489       0.158142
182151  29.986300 -98.775932       0.129439
182152  29.986300 -98.402489       0.116526
182952  30.311827 -99.149376       0.184782
182955  30.311827 -98.029045       0.155932
183754  30.638416 -99.149376       0.150101
184557  30.966091 -98.775932       0.124708
2022-09-27 0.1432411221

Data points used for interpolation:
         Latitude  Longitude  Soil_Moisture
182954  30.311827 -98.402489       0.094472
183755  30.638416 -98.775932       0.100915
183756  30.638416 -98.402489       0.126902
182151  29.986300 -98.775932       0.156890
182152  29.986300 -98.402489       0.160555
182955  30.311827 -98.029045       0.173721
183754  30.638416 -99.149376       0.090302
184557  30.966091 -98.775932       0.083824
2022-10-08 0.11968455854242221 Station Number: 0
Data points used for interpolation:
         Latitude  Longitude  Soil_Moisture
183755  30.638416 -98.775932       0.100915
183754  30.638416 -99.149376       0.090302
182954  30.311827 -98.402489       0.094472
182151  29.986300 -98.775932       0.156890
183756  30.638416 -98.402489       0.126902
184557  30.966091 -98.775932       0.083824
182150  29.986300 -99.149376       0.183129
182152  29.986300 -98.402489       0.160555
2022-10-08 0.11980735209297168 Station Number: 1
Data points used for interpolation:
  

Data points used for interpolation:
         Latitude  Longitude  Soil_Moisture
183755  30.638416 -98.775932       0.105916
183754  30.638416 -99.149376       0.096928
183756  30.638416 -98.402489       0.147531
184557  30.966091 -98.775932       0.138990
184556  30.966091 -99.149376       0.109989
184558  30.966091 -98.402489       0.163781
182951  30.311827 -99.522820       0.196063
183753  30.638416 -99.522820       0.109365
2022-10-22 0.12594741931003695 Station Number: 2
Data points used for interpolation:
         Latitude  Longitude  Soil_Moisture
183755  30.638416 -98.775932       0.105916
183754  30.638416 -99.149376       0.096928
184557  30.966091 -98.775932       0.138990
184556  30.966091 -99.149376       0.109989
183756  30.638416 -98.402489       0.147531
182951  30.311827 -99.522820       0.196063
183753  30.638416 -99.522820       0.109365
184558  30.966091 -98.402489       0.163781
2022-10-22 0.1251617028180063 Station Number: 3
Data points used for interpolation:
   

Data points used for interpolation:
         Latitude  Longitude  Soil_Moisture
182151  29.986300 -98.775932       0.168523
182954  30.311827 -98.402489       0.149044
183755  30.638416 -98.775932       0.120384
182952  30.311827 -99.149376       0.175911
182152  29.986300 -98.402489       0.168994
183756  30.638416 -98.402489       0.159177
182150  29.986300 -99.149376       0.188676
181349  29.661814 -98.775932       0.210038
2022-11-10 0.1641178128662442 Station Number: 5
./complete_satellite_data/ucar_cu_cygnss_sm_v1_2022_323.dap.nc4
Data points used for interpolation:
         Latitude  Longitude  Soil_Moisture
182151  29.986300 -98.775932       0.168388
182152  29.986300 -98.402489       0.161429
182955  30.311827 -98.029045       0.198625
184557  30.966091 -98.775932       0.177640
184558  30.966091 -98.402489       0.201306
183757  30.638416 -98.029045       0.206066
182150  29.986300 -99.149376       0.189499
182153  29.986300 -98.029045       0.206289
2022-11-19 0.18638672205

Data points used for interpolation:
         Latitude  Longitude  Soil_Moisture
184557  30.966091 -98.775932       0.166970
184558  30.966091 -98.402489       0.192837
183757  30.638416 -98.029045       0.188794
182150  29.986300 -99.149376       0.195332
184559  30.966091 -98.029045       0.183121
185359  31.294872 -98.775932       0.178236
185360  31.294872 -98.402489       0.196560
182956  30.311827 -97.655602       0.213896
2022-12-08 0.18837439497653913 Station Number: 0
Data points used for interpolation:
         Latitude  Longitude  Soil_Moisture
184557  30.966091 -98.775932       0.166970
182150  29.986300 -99.149376       0.195332
184558  30.966091 -98.402489       0.192837
183757  30.638416 -98.029045       0.188794
185359  31.294872 -98.775932       0.178236
184555  30.966091 -99.522820       0.207614
185358  31.294872 -99.149376       0.172812
184559  30.966091 -98.029045       0.183121
2022-12-08 0.18541001874634921 Station Number: 1
Data points used for interpolation:
  

In [7]:
for i, interp_df in enumerate(interpolation_dataframe_list):
    print("Station", i + 1)
    print(interp_df.isnull().sum(), "\n")

Station 1
Date             0
soil_moisture    3
distance         0
dtype: int64 

Station 2
Date             0
soil_moisture    3
distance         0
dtype: int64 

Station 3
Date             0
soil_moisture    3
distance         0
dtype: int64 

Station 4
Date             0
soil_moisture    3
distance         0
dtype: int64 

Station 5
Date             0
soil_moisture    3
distance         0
dtype: int64 

Station 6
Date             0
soil_moisture    3
distance         0
dtype: int64 



In [8]:
# Forward fill for remaining missing data
for interp_df in interpolation_dataframe_list:
    interp_df['soil_moisture'].fillna(method="ffill", inplace=True)
    
for i, interp_df in enumerate(interpolation_dataframe_list):
    print("Station", i + 1)
    print(interp_df.isnull().sum(), "\n")

Station 1
Date             0
soil_moisture    0
distance         0
dtype: int64 

Station 2
Date             0
soil_moisture    0
distance         0
dtype: int64 

Station 3
Date             0
soil_moisture    0
distance         0
dtype: int64 

Station 4
Date             0
soil_moisture    0
distance         0
dtype: int64 

Station 5
Date             0
soil_moisture    0
distance         0
dtype: int64 

Station 6
Date             0
soil_moisture    0
distance         0
dtype: int64 



/var/folders/dn/qw32n23j447c8wvwm7d77t400000gn/T/ipykernel_70010/1882530416.py:3: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  interp_df['soil_moisture'].fillna(method="ffill", inplace=True)


In [9]:
# Create new CSV files
for i in range(0,len(csv_filenames)):
    print("Station Number:", i)
    original_dataframe = pd.read_csv(csv_user_address + csv_filenames[i])
    interpolation_dataframe = interpolation_dataframe_list[i]

    original_dataframe['Date'] = pd.to_datetime(original_dataframe['Date'])
    interpolation_dataframe['Date'] = pd.to_datetime(interpolation_dataframe['Date'])

    # Merge the DataFrames on the 'Date' column, using a left join to keep all dates from df1
    merged_df = pd.merge(original_dataframe, interpolation_dataframe, on='Date', how='left', suffixes=('', '_from_df2'))

    # Drop the 'distance_from_df2' column as it's not needed (assuming all distances in df2 are 0 and not useful)
    merged_df.drop(columns=['distance_from_df2'], inplace=True)

    # If you want to update soil_moisture in df1 with values from df2 where available
    merged_df['soil_moisture'] = merged_df['soil_moisture'].fillna(merged_df['soil_moisture_from_df2'])

    # Drop the temporary 'soil_moisture_from_df2' column
    merged_df.drop(columns=['soil_moisture_from_df2'], inplace=True)
    merged_df_filename = "./satellite_data_csv/" + "Filled_Station" + str(i+1) + "_Satellite.csv"
    merged_df.to_csv(merged_df_filename, index=False)
    print(merged_df)

print(interpolation_dataframe_list)

Station Number: 0
           Date  soil_moisture   distance
0    2017-03-18       0.224324        NaN
1    2017-03-19       0.191104  12.266494
2    2017-03-20       0.210310  12.266494
3    2017-03-21       0.168200  12.266494
4    2017-03-22       0.167357        NaN
...         ...            ...        ...
2089 2022-12-06       0.158229  12.266494
2090 2022-12-07       0.161177  12.266494
2091 2022-12-08       0.188374        NaN
2092 2022-12-09       0.166370  12.266494
2093 2022-12-10       0.184179        NaN

[2094 rows x 3 columns]
Station Number: 1
           Date  soil_moisture  distance
0    2017-03-18       0.227771       NaN
1    2017-03-19       0.191104  9.984826
2    2017-03-20       0.210310  9.984826
3    2017-03-21       0.168200  9.984826
4    2017-03-22       0.166975       NaN
...         ...            ...       ...
2089 2022-12-06       0.158229  9.984826
2090 2022-12-07       0.161177  9.984826
2091 2022-12-08       0.185410       NaN
2092 2022-12-09       0.1